# Evaluations: the full loop

`grasp_agents.evals` evaluates any grasp-agents `Processor` — a single agent, a workflow, a parallel fan-out — against a versioned dataset, with per-example evaluators, aggregate metrics with confidence intervals, and immutable on-disk run records you can compare, rescore and push to Phoenix.

| concept | what it is |
|---|---|
| **Task** | the system under test: any `Processor` (or async function), run in an isolated session per trial |
| **Dataset** / **Example** | typed examples with stable ids, a reference (ground truth), metadata and splits |
| **Evaluator** → **Score** | a versioned instrument judging one trial (code or a validated LLM judge) |
| **Metric** | an aggregate over scores, with a 95% CI (repetitions are reduced per example first) |
| **Trial** | one execution of the task on one example (× repetitions) |
| **EvaluationRun** | the immutable record of a run: provenance, trials, scores, metrics |
| **Evaluation** | a named, importable definition — what the CLI and nightly jobs run |

The demo component is a **short-answer grader** for a tutoring product: a two-step `SequentialWorkflow` that checks which key points of a reference answer a student's answer covers, then writes a verdict and feedback. `v1` is a naive first attempt; `v2` handles paraphrases and negations. The writer flips borderline verdicts at random to behave like a sampled model. Everything runs offline; section 12 swaps in a real LLM.

The loop: **validate data → baseline → read failures → fix → compare → re-judge → A/B → held-out check → push**.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

from IPython.display import Markdown, display

DEMO_DIR = Path("../evals").resolve()
sys.path.insert(0, str(DEMO_DIR))
import grader_evals as demo  # the demo's pipeline, dataset, evaluators and evaluations

from grasp_agents.evals import (
    Dataset,
    LocalRunStore,
    compare,
    example_json_schema,
    pairwise,
    render_comparison_markdown,
    render_run_markdown,
)

# Runs normally go to ./.evals (or $GRASP_EVALS_DIR); keep the demo's apart.
EVALS_DIR = Path(tempfile.mkdtemp(prefix="grasp-evals-demo-"))
store = LocalRunStore(EVALS_DIR)
print("runs stored under", EVALS_DIR)

runs stored under /tmp/claude-501/grasp-evals-demo-evekzdf6


## 1. The component under test

A plain grasp-agents pipeline — nothing about it knows it is being evaluated.

In [2]:
grader = demo.build_grader("v1", noise=0.0)
submission = demo.Submission(
    question="What do plants need for photosynthesis?",
    reference_answer="Sunlight, water and carbon dioxide.",
    student_answer="Plants do not need sunlight, just water and carbon dioxide.",
)
packet = await grader.run(in_args=submission)
packet.payloads[0]

Grade(verdict='correct', feedback='Good job!')

## 2. The dataset

Typed examples (`Submission` in, the teacher's `TeacherGrade` as reference) with stable ids, `dev` / `test` splits and strata in metadata. The **fingerprint** hashes the content, so every run records exactly what it evaluated. `test` is **sealed**: reports show it only in aggregate, so whoever iterates on the grader — a person or a coding agent — cannot fit to its examples.

In [3]:
dataset = Dataset.load(demo.DATA, input_type=demo.Submission, reference_type=demo.TeacherGrade)
print(f"{len(dataset)} examples, fingerprint {dataset.fingerprint}")
print({split: len(dataset.split(split)) for split in dataset.splits})
example = dataset["bio-3"]
print(example.input.student_answer, "->", example.reference)

24 examples, fingerprint c7c7adb4b34d
{'dev': 16, 'test': 8}
Plants do not need sunlight, just water and carbon dioxide. -> verdict='incorrect' key_issue='sunlight'


Integrity checks run before every run (here: a correct answer cannot have a key issue). Agents authoring new examples get a JSON Schema for the record format:

In [4]:
print("problems:", demo.grader_v1.check_dataset(dataset))
schema = example_json_schema(demo.Submission, demo.TeacherGrade)
print(sorted(schema["properties"]), "required:", schema["required"])

problems: []
['id', 'input', 'metadata', 'reference', 'splits'] required: ['input']


## 3. Evaluators

Evaluators are versioned, task-specific instruments. Their outcomes are kept apart — a failure never becomes a number:

- a returned value is a **score** (`bool` = pass/fail, number, or `str` label);
- `Score.unscored(reason)` is **unscored** (e.g. an unparseable judge verdict) — excluded from metrics and counted;
- returning `None` means **not applicable** (here: the example has no key issue to name);
- an exception is an **evaluator failure**; a task exception or timeout is a **task error**.

In [5]:
import inspect

print(inspect.getsource(demo.names_key_issue._fn))

@evaluator(version="1")
def names_key_issue(ctx: Ctx) -> Score | None:
    """Does the feedback name what is wrong or missing? N/A when nothing is."""
    if ctx.reference is None or ctx.reference.key_issue is None:
        return None
    issue = ctx.reference.key_issue
    hit = issue.lower() in ctx.output.feedback.lower()
    return Score(
        name="names_key_issue",
        value=hit,
        explanation=f"expected '{issue}' in: {ctx.output.feedback}",
    )



## 4. Baseline

Each example runs **3 times** (the writer is noisy, like a sampled model). Metrics reduce repetitions per example first, so `n` counts examples, not trials; `pass^3` is the chance that all three runs agree with the teacher — the reliability a learner experiences.

In [6]:
run_v1 = await demo.grader_v1.run(split="dev", store=store)
display(Markdown(render_run_markdown(run_v1)))

# short-answer-grader — completed

- **Run:** `20261001T025747-short-answer-grader-931a` (evaluation) · 2026-10-01 02:57 UTC
- **Question:** Does the grader agree with teachers and explain mistakes?
- **Task:** grader@v1 (`grasp_agents.workflow.sequential_workflow.SequentialWorkflow`)
- **Dataset:** short_answers `c7c7adb4b34d` · 16/24 examples — split=dev
- **Evaluators:** agrees_with_teacher@1 (CODE), names_key_issue@1 (CODE), feedback_concise@1 (CODE), feedback_quality@1 (LLM)
- **Code:** `e5608163a110` (grasp-evals-lib)
- **Trials:** 48/48 (3 per example) · task errors 0 · evaluator failures 0 · unscored 0

## Metrics

| metric | value | 95% CI | n | missing | n/a |
|---|---|---|---|---|---|
| pass_rate(agrees_with_teacher) | 0.458 | [0.2, 0.717] | 16 | 0 | 0 |
|   ↳ difficulty=easy | 0.667 | [0.366, 0.967] | 11 | 0 | 0 |
|   ↳ difficulty=hard | 0 | [0, 0.434] | 5 | 0 | 0 |
| pass^3(agrees_with_teacher) | 0.375 | [0.109, 0.641] | 16 | 0 | 0 |
|   ↳ difficulty=easy | 0.545 | [0.195, 0.896] | 11 | 0 | 0 |
|   ↳ difficulty=hard | 0 | [0, 0] | 5 | 0 | 0 |
| pass_rate(names_key_issue) | 0 | [0, 0.324] | 8 | 0 | 8 |
|   ↳ difficulty=easy | 0 | [0, 0.561] | 3 | 0 | 8 |
|   ↳ difficulty=hard | 0 | [0, 0.434] | 5 | 0 | 0 |
| pass_rate(feedback_concise) | 1 | [0.806, 1] | 16 | 0 | 0 |
|   ↳ difficulty=easy | 1 | [0.741, 1] | 11 | 0 | 0 |
|   ↳ difficulty=hard | 1 | [0.566, 1] | 5 | 0 | 0 |
| pass_rate(feedback_quality) | 0.542 | [0.309, 0.774] | 16 | 0 | 0 |
|   ↳ difficulty=easy | 0.545 | [0.224, 0.866] | 11 | 0 | 0 |
|   ↳ difficulty=hard | 0.533 | [0.061, 1] | 5 | 0 | 0 |
| p95(duration_s) | 0.001 | [0.000551, 0.002] | 48 | 0 | 0 |
|   ↳ difficulty=easy | 0.001 | [0.000429, 0.002] | 33 | 0 | 0 |
|   ↳ difficulty=hard | 0.001 | [0.000475, 0.002] | 15 | 0 | 0 |

## Lowest `agrees_with_teacher`

| example | mean | reps | explanation (worst repetition) |
|---|---|---|---|
| `bio-2` | 0 | 3 | teacher: correct, grader: incorrect |
| `bio-3` | 0 | 3 | teacher: incorrect, grader: correct |
| `phy-2` | 0 | 3 | teacher: correct, grader: partial |
| `phy-3` | 0 | 3 | teacher: incorrect, grader: partial |
| `phy-4` | 0 | 3 | teacher: incorrect, grader: correct |

## Lowest `feedback_quality`

| example | mean | reps | explanation (worst repetition) |
|---|---|---|---|
| `bio-1` | 0 | 3 | Good job! |
| `bio-3` | 0 | 3 | Good job! |
| `phy-1` | 0 | 3 | Good job! |
| `chem-1` | 0 | 3 | Good job! |
| `chem-2` | 0 | 3 | Good job! |

## Lowest `names_key_issue`

| example | mean | reps | explanation (worst repetition) |
|---|---|---|---|
| `bio-3` | 0 | 3 | expected 'sunlight' in: Good job! |
| `bio-4` | 0 | 3 | expected 'carbon' in: Some points are missing. |
| `phy-3` | 0 | 3 | expected 'gravity' in: Some points are missing. |
| `phy-4` | 0 | 3 | expected 'attract' in: Good job! |
| `math-3` | 0 | 3 | expected 'half' in: Some points are missing. |


## 5. Read the failures

The headline number is only a pointer; the examples behind it are the evidence.

In [7]:
for trial in run_v1.trials:
    agree = trial.score("agrees_with_teacher")
    if trial.repetition == 0 and agree is not None and agree.value is False:
        student = run_v1.example(trial.example_id).input.student_answer
        print(f"{trial.example_id:7} {agree.explanation:38} | {student}")

bio-2   teacher: correct, grader: incorrect    | They need sun, H2O and CO2.
bio-3   teacher: incorrect, grader: correct    | Plants do not need sunlight, just water and carbon dioxide.
phy-2   teacher: correct, grader: partial      | Because gravity pulls things down to earth.
phy-3   teacher: incorrect, grader: partial    | Objects fall because the Earth spins.
phy-4   teacher: incorrect, grader: correct    | Gravity does not attract objects; they just fall toward Earth.
math-2  teacher: correct, grader: incorrect    | A half, since each half piece is bigger than a third.
math-3  teacher: incorrect, grader: partial    | One third is larger because three pieces are more than two.
chem-3  teacher: incorrect, grader: correct    | It is made of hydrogen and carbon.


v1 misses paraphrases ("sun, H2O and CO2") and negations ("do not need sunlight"), and its feedback never names the problem. `v2` adds synonyms, stemming and negation handling.

## 6. Fix, rerun, compare

The comparison pairs examples by id (and unchanged content) and reports the **paired** difference with a CI and p-value, the **minimum detectable effect** for this dataset size, and the examples that moved.

In [8]:
run_v2 = await demo.grader_v2.run(split="dev", store=store)
display(Markdown(render_comparison_markdown(compare(run_v1, run_v2))))

# short-answer-grader vs short-answer-grader

- **Base:** `20261001T025747-short-answer-grader-931a` · **Candidate:** `20261001T025747-short-answer-grader-1611`
- **Paired examples:** 16 (changed content 0, base-only 0, candidate-only 0)

Δ = candidate - base on paired examples; ✱ = 95% CI excludes 0; MDE = smallest true Δ detectable at 80% power.

| target | base | candidate | Δ | 95% CI | p | MDE | n | +/- |
|---|---|---|---|---|---|---|---|---|
| ↑ agrees_with_teacher | 0.458 | 0.708 | 0.25 ✱ | [0.03, 0.47] | 0.029 | 0.289 | 16 | +5/-0 |
| ↑ feedback_concise | 1 | 1 | 0 | [0, 0] | 1 | 0 | 16 | +0/-0 |
| ↑ feedback_quality | 0.542 | 1 | 0.458 ✱ | [0.226, 0.691] | 0.000776 | 0.306 | 16 | +10/-0 |
| ↑ names_key_issue | 0 | 0.917 | 0.917 ✱ | [0.788, 1] | 6.47e-07 | 0.153 | 8 | +8/-0 |
| ↓ error | 0 | 0 | 0 | [0, 0] | 1 | 0 | 16 | +0/-0 |
| ↓ duration_s | 0.000498 | 0.00046 | -3.81e-05 | [-9.28e-05, 1.66e-05] | 0.159 | 7.19e-05 | 16 | +9/-7 |


## 7. Change the instrument, not the task: rescoring

A stricter `feedback_quality` judge (version 2) re-scores v2's **stored outputs** — the task does not run again. The result is a child run; the parent is never modified, and the comparison warns that the evaluator version changed.

In [9]:
strict = await demo.grader_v2_strict.rescore(run_v2, store=store)
print("child of", strict.parent_run_id)
display(Markdown(render_comparison_markdown(compare(run_v2, strict))))

child of 20261001T025747-short-answer-grader-1611


# short-answer-grader vs short-answer-grader

- **Base:** `20261001T025747-short-answer-grader-1611` · **Candidate:** `20261001T025747-short-answer-grader-83f8`
- **Paired examples:** 16 (changed content 0, base-only 0, candidate-only 0)
- ⚠ Evaluator 'feedback_quality' changed version (1 → 2): score differences may come from the instrument.

Δ = candidate - base on paired examples; ✱ = 95% CI excludes 0; MDE = smallest true Δ detectable at 80% power.

| target | base | candidate | Δ | 95% CI | p | MDE | n | +/- |
|---|---|---|---|---|---|---|---|---|
| ↑ agrees_with_teacher | 0.708 | 0.708 | 0 | [0, 0] | 1 | 0 | 16 | +0/-0 |
| ↑ feedback_concise | 1 | 1 | 0 | [0, 0] | 1 | 0 | 16 | +0/-0 |
| ↑ feedback_quality | 1 | 0.958 | -0.042 | [-0.102, 0.019] | 0.164 | 0.08 | 16 | +0/-2 |
| ↑ names_key_issue | 0.917 | 0.917 | 0 | [0, 0] | 1 | 0 | 8 | +0/-0 |
| ↓ error | 0 | 0 | 0 | [0, 0] | 1 | 0 | 16 | +0/-0 |
| ↓ duration_s | 0.00046 | 0.00046 | 0 | [0, 0] | 1 | 0 | 16 | +0/-0 |

## Regressions in `feedback_quality`

| example | base | candidate | candidate explanation |
|---|---|---|---|
| `bio-4` | 1 | 0.667 | mentions a key point: True — You missed: carbon, dioxide. |
| `chem-3` | 1 | 0.667 | mentions a key point: False — Correct and complete. |


## 8. Pairwise A/B

A pairwise judge sees two outputs as "first" and "second" — never which version produced them — and is asked in **both orders**, so position bias shows up as inconsistency. The result is itself a run, with a win rate and CI.

In [10]:
ab = await pairwise(
    run_v1,
    run_v2,
    demo.specific_feedback_judge,
    input_type=demo.Submission,
    reference_type=demo.TeacherGrade,
    output_type=demo.Grade,
    store=store,
)
win = ab.metric("win_rate(specificity)")
print(f"v2 wins {win.value:.0%} [{win.ci_low:.0%}, {win.ci_high:.0%}], details: {win.details}")
print("position-consistent:", ab.metric("pass_rate(specificity.position_consistent)").value)

v2 wins 75% [63%, 87%], details: {'candidate': 24, 'base': 0, 'tie': 24, 'inconsistent': 0, 'sign_test_p': 1.1920928955078125e-07}
position-consistent: 1.0


## 9. The held-out test split

Only aggregates are shown for sealed splits — no per-example tables.

In [11]:
run_test = await demo.grader_v2.run(split="test", store=store)
display(Markdown(render_run_markdown(run_test)))

# short-answer-grader — completed

- **Run:** `20261001T025747-short-answer-grader-287c` (evaluation) · 2026-10-01 02:57 UTC
- **Question:** Does the grader agree with teachers and explain mistakes?
- **Task:** grader@v2 (`grasp_agents.workflow.sequential_workflow.SequentialWorkflow`)
- **Dataset:** short_answers `c7c7adb4b34d` · 8/24 examples — split=test
- **Evaluators:** agrees_with_teacher@1 (CODE), names_key_issue@1 (CODE), feedback_concise@1 (CODE), feedback_quality@1 (LLM)
- **Code:** `e5608163a110` (grasp-evals-lib)
- **Trials:** 24/24 (3 per example) · task errors 0 · evaluator failures 0 · unscored 0
- **Sealed:** 24 trials in held-out splits are reported in aggregate only

## Metrics

| metric | value | 95% CI | n | missing | n/a |
|---|---|---|---|---|---|
| pass_rate(agrees_with_teacher) | 0.875 | [0.529, 0.978] | 8 | 0 | 0 |
|   ↳ difficulty=easy | 1 | [0.51, 1] | 4 | 0 | 0 |
|   ↳ difficulty=hard | 0.75 | [0.301, 0.954] | 4 | 0 | 0 |
| pass^3(agrees_with_teacher) | 0.875 | [0.579, 1] | 8 | 0 | 0 |
|   ↳ difficulty=easy | 1 | [1, 1] | 4 | 0 | 0 |
|   ↳ difficulty=hard | 0.75 | [0, 1] | 4 | 0 | 0 |
| pass_rate(names_key_issue) | 1 | [0.439, 1] | 3 | 0 | 5 |
|   ↳ difficulty=easy | - | - | 0 | 0 | 4 |
|   ↳ difficulty=hard | 1 | [0.439, 1] | 3 | 0 | 1 |
| pass_rate(feedback_concise) | 1 | [0.676, 1] | 8 | 0 | 0 |
|   ↳ difficulty=easy | 1 | [0.51, 1] | 4 | 0 | 0 |
|   ↳ difficulty=hard | 1 | [0.51, 1] | 4 | 0 | 0 |
| pass_rate(feedback_quality) | 1 | [0.676, 1] | 8 | 0 | 0 |
|   ↳ difficulty=easy | 1 | [0.51, 1] | 4 | 0 | 0 |
|   ↳ difficulty=hard | 1 | [0.51, 1] | 4 | 0 | 0 |
| p95(duration_s) | 0.001 | [0.000606, 0.002] | 24 | 0 | 0 |
|   ↳ difficulty=easy | 0.001 | [0.000572, 0.001] | 12 | 0 | 0 |
|   ↳ difficulty=hard | 0.002 | [0.000492, 0.002] | 12 | 0 | 0 |


## 10. The same loop from the shell

Everything above is also a CLI (`grasp-evals`, or `python -m grasp_agents.evals`) with `--json` output and gating exit codes — the interface a coding agent uses. `walkthrough.sh` next to the demo runs the whole loop.

In [12]:
import json
import subprocess


def evals(*args: str) -> subprocess.CompletedProcess[str]:
    env = {**os.environ, "GRASP_EVALS_DIR": str(EVALS_DIR)}
    return subprocess.run(
        [sys.executable, "-m", "grasp_agents.evals", *args],
        capture_output=True, text=True, env=env,
    )


print(evals("runs").stdout)

# What an agent runs in its loop: a small, fast slice, machine-readable, gated.
SPEC = f"{DEMO_DIR / 'grader_evals.py'}:grader_v2"
result = evals(
    "run", SPEC, "--split", "dev", "--limit", "4", "-r", "1", "-q", "--json",
    "--fail-under", "pass_rate(agrees_with_teacher)=0.5",
)
summary = json.loads(result.stdout)
print("exit code:", result.returncode, "| gate failures:", summary["gate_failures"])
print(json.dumps(summary["metrics"]["pass_rate(agrees_with_teacher)"], indent=2))

20261001T025747-short-answer-grader-287c  completed  24 trials  pass_rate(agrees_with_teacher)=0.875
20261001T025747-short-answer-grader-vs-short-answer-grad-f2f1  completed  48 trials  win_rate(specificity)=0.75
20261001T025747-short-answer-grader-83f8  completed  48 trials  pass_rate(agrees_with_teacher)=0.708
20261001T025747-short-answer-grader-1611  completed  48 trials  pass_rate(agrees_with_teacher)=0.708
20261001T025747-short-answer-grader-931a  completed  48 trials  pass_rate(agrees_with_teacher)=0.458



exit code: 0 | gate failures: []
{
  "value": 0.5,
  "ci": [
    0.15003898915214958,
    0.8499610108478504
  ],
  "n": 4,
  "missing": 0,
  "groups": {
    "difficulty=easy": {
      "value": 0.3333333333333333,
      "ci": [
        0.06149194472039626,
        0.7923403991979521
      ],
      "n": 3,
      "missing": 0
    },
    "difficulty=hard": {
      "value": 1.0,
      "ci": [
        0.2065493143772375,
        1.0
      ],
      "n": 1,
      "missing": 0
    }
  }
}


## 11. Phoenix (optional)

Runs stay canonical on disk; Phoenix is the shared store for datasets and the UI. A finished run is pushed as one experiment (one Phoenix run per trial, one evaluation per score); pushing again adds nothing. Set `PHOENIX_BASE_URL` (and `PHOENIX_API_KEY`) to try it.

In [13]:
if os.environ.get("PHOENIX_BASE_URL"):
    from grasp_agents.evals.phoenix import PhoenixClient, experiment_url, push_run

    async with PhoenixClient() as client:
        link = await push_run(client, run_v2, store=store)
    print("experiment:", experiment_url(link))
else:
    print("PHOENIX_BASE_URL not set; skipping the push")

experiment: http://127.0.0.1:6116/datasets/RGF0YXNldDoxMA==/compare?experimentId=RXhwZXJpbWVudDo4


## 12. Swap in a real model

The task is any `Processor`, so an `LLMAgent` drops in unchanged; per-agent token usage, cost and the models observed are recorded on every trial. Use a cheap model.

In [14]:
if os.environ.get("RUN_REAL_LLM"):
    from grasp_agents.llm_providers.gemini.gemini_llm import GeminiLLM

    llm = GeminiLLM(model_name="gemini-3.5-flash-lite", apply_output_schema_via_provider=True)
    llm_evaluation = demo.llm_grader_evaluation(llm)
    run_llm = await llm_evaluation.run(split="dev", repetitions=1, store=store)
    display(Markdown(render_run_markdown(run_llm)))
else:
    print("set RUN_REAL_LLM=1 (and a Gemini API key) to run the LLM grader")

set RUN_REAL_LLM=1 (and a Gemini API key) to run the LLM grader
